In [ ]:
# Colab Cell 1 — Install dependencies
!pip install --upgrade pip >/dev/null
!pip install azure-storage-blob mysql-connector-python requests tqdm python-dotenv Pillow >/dev/null
!pip install openai


In [ ]:
# Colab Cell 2 — MySQL connection and table setup
import os
from datetime import datetime
import mysql.connector
from mysql.connector import errorcode

MYSQL_HOST = os.getenv("MYSQL_HOST", "cycleai.mysql.database.azure.com")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "admindev")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "Zana81333")
MYSQL_DB = os.getenv("MYSQL_DB", "cycle_ai")

def get_mysql_conn():
    cfg = {
        "host": MYSQL_HOST,
        "port": MYSQL_PORT,
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "database": MYSQL_DB,
        "autocommit": False,
    }
    try:
        conn = mysql.connector.connect(**cfg)
        return conn
    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Something is wrong with your user name or password")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist")
            raise

print("MySQL connected and table ensured: perception_poll_votes_ai")


In [ ]:
# Colab Cell 3 — Azure Blob connection and caching blob names
import os, io, random
from azure.storage.blob import BlobServiceClient
from PIL import Image

AZURE_CONN_STR = os.getenv("AZURE_CONN_STR", "")   # recommended: store in Colab secret or env
BLOB_CONTAINER = os.getenv("BLOB_CONTAINER", "london")

if not AZURE_CONN_STR:
    raise RuntimeError("AZURE_CONN_STR not set. Set as environment variable or inline in this cell.")

blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

print("Listing blobs in 'Images/' folder (this may take a few seconds)...")

IMAGE_PREFIX = "training/training/Images/"
all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

blob_names = [
    blob.name for blob in all_image_blobs
    if blob.name.lower().endswith(IMAGE_EXTENSIONS)
]

print(f"Found {len(blob_names)} image files.")
if len(blob_names) < 2:
    raise RuntimeError(f"Need at least 2 blobs in container to sample pairs. Found: {len(blob_names)}")

def download_blob_to_pil(blob_name, timeout=30):
    """Download a blob and return it as a PIL Image."""
    blob_data = container_client.download_blob(blob_name, timeout=timeout).readall()
    img = Image.open(io.BytesIO(blob_data)).convert("RGB")
    return img

print("Azure blob helpers ready.")


In [ ]:
# Colab Cell: Check for Rapid/Spam Voting
import pandas as pd
import mysql.connector
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

conn = get_mysql_conn()

sql_spam_check = """
SELECT DISTINCT
    t1.user_id
FROM perception_poll_votes_london t1
JOIN perception_poll_votes_london t2
  ON t1.user_id = t2.user_id
  AND t2.voting_time > t1.voting_time
  AND t2.voting_time <= t1.voting_time + INTERVAL 1 SECOND;
"""

try:
    print("Checking for duplicate votes on the same pair...")
    df_spam = pd.read_sql(sql_spam_check, conn)

    if not df_spam.empty:
        print(f"\n⚠️ FOUND {len(df_spam)} DUPLICATE VOTES")
        print("These users voted on the same pair twice within 1 second:")

        pd.set_option('display.max_rows', 50)
        display(df_spam)

        print("\n--- Offenders Summary ---")
        summary = df_spam.groupby('user_id').size().reset_index(name='duplicate_count')
        display(summary.sort_values('duplicate_count', ascending=False))
        unreliable_user_uuids = df_spam['user_id'].tolist()
    else:
        print("\n✅ Clean. No users found voting on the same pair multiple times within 2 seconds.")

except Exception as e:
    print(f"Error running check: {e}")
finally:
    conn.close()
    pass

In [ ]:
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"

In [ ]:
# Colab Cell 4 (Modified for GPT-4o + Database Source + Comparison)
import os, io, json, time, base64, hashlib, random, re
from tqdm import tqdm
from PIL import Image
import requests
import mysql.connector
from datetime import datetime
from time import sleep
import uuid
from IPython.display import display, HTML
import pandas as pd
import openai
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ---------- CONFIG ----------
OPENAI_MODEL_ID = "gpt-5"
client = openai.OpenAI(api_key=OPENAI_API_KEY)

WORKER_ID = os.getenv("WORKER_ID", str(uuid.uuid4()))

# How many rows to fetch from the London votes table
N=40
BATCH_COMMIT = 5
RATE_LIMIT_SECONDS = float(os.getenv("RATE_LIMIT_SECONDS", "0.1"))
RETRY_MAX = 3
IMAGE_PREFIX = "training/training/Images/" # Prefix to add to DB filenames

FULL_PROMPT_TEXT = (
    "You will see two images: Image A (first) and Image B (second).\n\n"
    "TASK: Decide which image looks SAFER for cyclists.\n\n"
    "Respond with ONE line only in the exact required format:\n"
    "[VOTE]: [Vote Number] - [reason]\n\n"
    "Where Vote Number is:\n"
    "  0 - Both look equally safe\n"
    "  1 - Image A looks safer\n"
    "  2 - Image B looks safer\n\n"
    "Keep your reason concise. DO NOT include any extra text.\n\n"
    "Examples (Use this format EXACTLY):\n"
    "[VOTE]: 1 - Image A has a dedicated bike lane.\n"
    "[VOTE]: 2 - Image B has fewer vehicles present.\n"
    "[VOTE]: 0 - Both show similar traffic conditions.\n"
    "\nNow decide which image looks safer."
)

# Stats tracking
results = []
agreement_stats = {"matches": 0, "total": 0}

def pil_to_data_uri(pil_img, quality=85):
    b64 = pil_to_base64_jpeg(pil_img, quality=quality)
    return f"data:image/jpeg;base64,{b64}"

def display_pair(pil1, pil2, ai_vote, human_vote, reason, name1, name2, max_width=320):
    uri1 = pil_to_data_uri(pil1)
    uri2 = pil_to_data_uri(pil2)

    # Color code match status
    match_color = "green" if ai_vote == human_vote else "red"
    match_text = "MATCH" if ai_vote == human_vote else "DIFF"

    caption = (
        f"<b>AI Vote:</b> {ai_vote} | <b>Human Vote:</b> {human_vote} "
        f"(<span style='color:{match_color};font-weight:bold'>{match_text}</span>)<br>"
        f"<b>Reason:</b> {reason}<br>"
        f"<b>Files:</b> {name1} — {name2}"
    )

    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>A: {name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>B: {name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))

# ---------- UTILITIES ----------
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format="JPEG", quality=quality)
    return base64.b64encode(buff.getvalue()).decode("utf-8")

def idempotency_key(name1, name2):
    s = f"{name1}|{name2}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def upsert_vote(conn, rec):
    cur = conn.cursor()
    sql = """
    INSERT INTO perception_poll_votes_ai
      (image_name1, image_name2, vote, model_id, user_id, reason)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = (rec["image_name1"], rec["image_name2"], rec["vote"], rec["model_id"], rec["user_id"], rec.get("reason"))
    cur.execute(sql, params)
    cur.close()

def get_source_votes(conn, limit):
    """Fetch pairs and existing human votes from the source table."""
    cur = conn.cursor(dictionary=True)
    uuid_str = ', '.join(f"'{u}'" for u in unreliable_user_uuids)
    sql = f"""
        SELECT image_name1, image_name2, vote
        FROM perception_poll_votes_london
        WHERE user_id NOT IN ({uuid_str})
        ORDER BY RAND()
        LIMIT {limit}
    """
    cur.execute(sql)
    rows = cur.fetchall()
    cur.close()
    return rows


def call_model_openai(prompt_text, pil1, pil2):
    try:
        uri1 = pil_to_data_uri(pil1)
        uri2 = pil_to_data_uri(pil2)

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Image A:"},
                    {"type": "image_url", "image_url": {"url": uri1}},
                    {"type": "text", "text": "\nImage B:"},
                    {"type": "image_url", "image_url": {"url": uri2}},
                    {"type": "text", "text": f"\n\n{prompt_text}"}
                ]
            }
        ]

        response = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_completion_tokens=1064
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        raise

# ---------- MAIN LOOP ----------
print(f"Starting comparison loop. Target N={N}, Model={OPENAI_MODEL_ID}")
conn = get_mysql_conn()
insert_count = 0

# 1. Fetch Source Data
print("Fetching rows from 'perception_poll_votes_london'...")
source_rows = get_source_votes(conn, N)
print(f"Fetched {len(source_rows)} rows to process.")

# Ensure Blob Client is ready
try:
    blob_svc
except NameError:
    from azure.storage.blob import BlobServiceClient
    blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)

pbar = tqdm(source_rows, desc="Processing Pairs")

for row in pbar:
    # Extract names from DB
    clean_name1 = row['image_name1']
    clean_name2 = row['image_name2']
    human_vote = int(row['vote'])

    # Add prefix for Blob Storage retrieval
    blob_name1 = IMAGE_PREFIX + clean_name1
    blob_name2 = IMAGE_PREFIX + clean_name2

    pil1 = pil2 = None

    # Download Images
    download_success = False
    for attempt in range(RETRY_MAX):
        try:
            pil1 = download_blob_to_pil(blob_name1)
            pil2 = download_blob_to_pil(blob_name2)
            download_success = True
            break
        except Exception as e:
            wait = 2 ** attempt
            time.sleep(wait)

    if not download_success or pil1 is None or pil2 is None:
        pbar.set_postfix(status="img_fail")
        continue

    # AI Inference
    try:
        raw_txt = call_model_openai(FULL_PROMPT_TEXT, pil1, pil2)
    except Exception as e:
        raw_txt = f"ERROR: {str(e)}"

    # Check for explicit errors we caught above
    if str(raw_txt).startswith("ERROR:"):
        print(f"Skipping {clean_name1}: {raw_txt}")
        pbar.set_postfix(status="err_skip")
        continue

    # Parse Result
    match = re.search(r"^(?:\[VOTE\]\s*[:\-]\s*)?\s*([0-2])\s*[:\-]\s*(.*)$", str(raw_txt).strip(), re.MULTILINE | re.IGNORECASE)

    if not match:
        # Now we can see the actual text if it's not empty
        print(f"\nParse error for {clean_name1}. Raw content: '{raw_txt}'")
        continue

    ai_vote_val = int(match.group(1))
    reason_text = match.group(2).strip()

    # Compare Votes
    agreement_stats["total"] += 1
    if ai_vote_val == human_vote:
        agreement_stats["matches"] += 1

    # Prepare Record for AI Table
    rec = {
        "image_name1": clean_name1,
        "image_name2": clean_name2,
        "vote": ai_vote_val,
        "model_id": OPENAI_MODEL_ID,
        "user_id": WORKER_ID,
        "reason": reason_text,
    }

    # Insert into DB
    try:
        upsert_vote(conn, rec)
        insert_count += 1
    except Exception as e:
        print("DB insert error:", e)
        conn.rollback()
        continue

    # Store result for dataframe display
    results.append({
        "image1": clean_name1,
        "image2": clean_name2,
        "human_vote": human_vote,
        "ai_vote": ai_vote_val,
        "match": (human_vote == ai_vote_val),
        "reason": reason_text
    })

    # Display (Optional, can comment out for speed)
    try:
        display_pair(pil1, pil2, ai_vote_val, human_vote, reason_text, clean_name1, clean_name2)
    except Exception:
        pass

    if insert_count % BATCH_COMMIT == 0:
        try:
            conn.commit()
        except Exception:
            conn.rollback()

    pbar.set_postfix(match_rate=f"{agreement_stats['matches']}/{agreement_stats['total']}")
    time.sleep(RATE_LIMIT_SECONDS)

# Final Commit
try:
    conn.commit()
except Exception:
    conn.rollback()
conn.close()

# Final Stats
print("\n" + "="*40)
print("PROCESSING COMPLETE")
print("="*40)
print(f"Total Pairs Processed: {agreement_stats['total']}")
print(f"Identical Votes:       {agreement_stats['matches']}")
if agreement_stats['total'] > 0:
    acc = (agreement_stats['matches'] / agreement_stats['total']) * 100
    print(f"Agreement Rate:        {acc:.2f}%")
print(f"Records inserted:      {insert_count}")

if results:
    df = pd.DataFrame(results)
    cols = ["image1", "image2", "human_vote", "ai_vote", "match", "reason"]
    display(df[cols].head(50))
    print("\n" + "="*40)
    print("CONFUSION MATRIX & REPORT")
    print("="*40)

    y_true = [r['human_vote'] for r in results]
    y_pred = [r['ai_vote'] for r in results]

    labels = [0, 1, 2]
    label_names = ['Equal (0)', 'Img A (1)', 'Img B (2)']

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, labels=labels, target_names=label_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True,
                xticklabels=label_names,
                yticklabels=label_names)
    plt.xlabel('AI Prediction')
    plt.ylabel('Human Truth')
    plt.title(f'Confusion Matrix (Model: {OPENAI_MODEL_ID})')
    plt.show()
else:
    print("No results generated.")